In [2]:
import requests
import pandas as pd
import duckdb
from datetime import datetime

# -----------------------------
# CONNECT TO DATABASE
# -----------------------------
conn = duckdb.connect(
    "../database/airport_ops.duckdb"
)

# -----------------------------
# GET LIVE FLIGHTS
# -----------------------------
url = (
    "https://opensky-network.org"
    "/api/states/all"
)

# no username/password
response = requests.get(url)

print("Status:", response.status_code)

data = response.json()

# -----------------------------
# COLUMN NAMES
# -----------------------------
columns = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source"
]

# -----------------------------
# CREATE DATAFRAME
# -----------------------------
flights = pd.DataFrame(
    data["states"],
    columns=columns
)

# add timestamp
flights["retrieved_at"] = (
    datetime.utcnow()
)

print(
    f"Retrieved "
    f"{len(flights)} flights"
)

# preview
print(flights.head())

# -----------------------------
# SAVE TO DATABASE
# -----------------------------
conn.register(
    "flights_df",
    flights
)

conn.execute("""
CREATE OR REPLACE TABLE flights AS
SELECT *
FROM flights_df
""")

print("Flights table created!")

conn.close()

Status: 200
Retrieved 11963 flights
   icao24  callsign origin_country  time_position  last_contact  longitude  \
0  39de4e  TVF35DL          France   1.779297e+09    1779296619     2.4083   
1  801638  AXB2381           India   1.779297e+09    1779296618    75.6808   
2  39de4a  TVF39PQ          France   1.779297e+09    1779296619     6.5812   
3  39de4d  TVF50AD          France   1.779297e+09    1779296619     7.4734   
4  aa9321  UAL130    United States   1.779297e+09    1779296619  -120.9654   

   latitude  baro_altitude  on_ground  velocity  true_track  vertical_rate  \
0   48.7286          45.72      False     68.90      254.41          -2.93   
1   27.4946        9189.72      False    249.28       24.77         -11.05   
2   46.5071       11574.78      False    210.35      298.01           0.00   
3   46.1166       10972.80      False    208.75      300.67           0.00   
4   54.1354       10668.00      False    277.28      103.08           0.00   

  sensors  geo_altitude sq

/tmp/ipykernel_27650/270803467.py:61: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow()
